In [1]:
# Import required libraries

from pathlib import Path
from collections import Counter
import polars as pl

In [2]:
# Define project data paths

PROJECT_ROOT = Path.cwd().parents[1]

DATA_ROOT = PROJECT_ROOT / "tennis_data"

EXTRACT_ROOT = DATA_ROOT / "extracted"

In [5]:
# Collect all home_team_score parquet files

home_team_score_files = sorted(
    EXTRACT_ROOT.glob(
        "*/raw_match_parquet/home_team_score_*.parquet"
    )
)

print(
    "Number of home_team_score files:",
    len(home_team_score_files)
)

Number of home_team_score files: 35164


In [6]:
# Display a few file names for validation

for file in home_team_score_files[:10]:
    print(file.name)

home_team_score_11974053.parquet
home_team_score_11974066.parquet
home_team_score_11998445.parquet
home_team_score_11998446.parquet
home_team_score_11998447.parquet
home_team_score_11998448.parquet
home_team_score_11998449.parquet
home_team_score_11998450.parquet
home_team_score_11998451.parquet
home_team_score_11998456.parquet


In [7]:
# Check column-count consistency across all home_team_score files

column_count_distribution = Counter(
    len(pl.read_parquet(file).columns)
    for file in home_team_score_files
)

print("Column-count distribution:")

for column_count, file_count in sorted(
    column_count_distribution.items()
):
    print(
        f"{column_count} columns: {file_count} files"
    )

Column-count distribution:
14 columns: 35164 files


In [9]:
# Check whether all home_team_score files have identical column names

reference_columns = pl.read_parquet(
    home_team_score_files[0]
).columns

different_column_files = []

for file in home_team_score_files:
    columns = pl.read_parquet(file).columns

    if columns != reference_columns:
        different_column_files.append(file)

print(
    "Files with different column names:",
    len(different_column_files)
)

Files with different column names: 0


In [10]:
# Check whether home_team_score files have different schemas

reference_schema = pl.read_parquet(
    home_team_score_files[0]
).schema

different_schema_files = []

for file in home_team_score_files:
    schema = pl.read_parquet(file).schema

    if schema != reference_schema:
        different_schema_files.append(file)

print(
    "Files with different schema:",
    len(different_schema_files)
)

Files with different schema: 35112


In [11]:
# Show all observed data types for each column

column_dtypes = {
    column: set()
    for column in reference_columns
}

for file in home_team_score_files:
    schema = pl.read_parquet(file).schema

    for column, dtype in schema.items():
        column_dtypes[column].add(str(dtype))

for column, dtypes in column_dtypes.items():
    print(
        f"{column}: {sorted(dtypes)}"
    )

match_id: ['Int64']
current_score: ['Int64', 'Null']
display_score: ['Int64', 'Null']
period_1: ['Int64', 'Null']
period_2: ['Int64', 'Null']
period_3: ['Int64', 'Null']
period_4: ['Null']
period_5: ['Null']
period_1_tie_break: ['Int64', 'Null']
period_2_tie_break: ['Int64', 'Null']
period_3_tie_break: ['Int64', 'Null']
period_4_tie_break: ['Null']
period_5_tie_break: ['Null']
normal_time: ['Null']


In [12]:
# Define the standard schema for home_team_score

target_schema = {
    "match_id": pl.Int64,
    "current_score": pl.Int64,
    "display_score": pl.Int64,
    "period_1": pl.Int64,
    "period_2": pl.Int64,
    "period_3": pl.Int64,
    "period_4": pl.Int64,
    "period_5": pl.Int64,
    "period_1_tie_break": pl.Int64,
    "period_2_tie_break": pl.Int64,
    "period_3_tie_break": pl.Int64,
    "period_4_tie_break": pl.Int64,
    "period_5_tie_break": pl.Int64,
    "normal_time": pl.Int64,
}

In [15]:
# Read all home_team_score files with a fixed schema
# and add the snapshot date from the parent folder

home_team_score_frames = []

for file in home_team_score_files:
    df = pl.read_parquet(file)

    df = df.cast(
        target_schema,
        strict=False
    )

    snapshot_date = file.parent.parent.name

    df = df.with_columns(
        pl.lit(snapshot_date)
        .str.to_date("%Y%m%d")
        .alias("snapshot_date")
    )

    home_team_score_frames.append(df)

print(
    "Processed away_team_score files:",
    len(home_team_score_frames)
)

Processed away_team_score files: 35164


In [16]:
# Concatenate all processed home_team_score snapshots

home_team_score = pl.concat(
    home_team_score_frames,
    how="vertical_relaxed"
)

print(
    "Final home_team_score shape:",
    home_team_score.shape
)

Final home_team_score shape: (35164, 15)


In [17]:
# Check the number of snapshots for each match

snapshot_counts = (
    home_team_score
    .group_by("match_id")
    .len()
    .rename({"len": "snapshot_count"})
)

snapshot_counts.group_by("snapshot_count").len().sort("snapshot_count")

snapshot_count,len
u32,u32
1,896
2,13676
3,2288
4,13


In [18]:
# Check the maximum number of snapshots for a match

snapshot_counts.select(
    pl.col("snapshot_count").max().alias("max_snapshots")
)

max_snapshots
u32
4


In [19]:
# Find some matches with multiple snapshots

snapshot_counts.filter(
    pl.col("snapshot_count") > 1
).sort(
    "snapshot_count",
    descending=True
).head(10)

match_id,snapshot_count
i64,u32
12063587,4
12063599,4
12063611,4
12084420,4
12063582,4
12063588,4
12086016,4
12102168,4
12063615,4


In [21]:
# Check how many matches have different score values across snapshots

score_columns = [
    "current_score",
    "display_score",
    "period_1",
    "period_2",
    "period_3",
    "period_4",
    "period_5",
    "period_1_tie_break",
    "period_2_tie_break",
    "period_3_tie_break",
    "period_4_tie_break",
    "period_5_tie_break",
    "normal_time",
]

score_changes = (
    home_team_score
    .group_by("match_id")
    .agg([
        pl.struct(score_columns).n_unique().alias("unique_score_versions"),
        pl.len().alias("snapshot_count"),
    ])
)

score_changes.group_by("unique_score_versions").len().sort("unique_score_versions")

unique_score_versions,len
u32,u32
1,12460
2,4153
3,260


In [22]:
# Get one match with 3 different score versions

sample_match_id = (
    score_changes
    .filter(pl.col("unique_score_versions") == 3)
    .select("match_id")
    .item(0, 0)
)

print("Sample match_id:", sample_match_id)

home_team_score.filter(
    pl.col("match_id") == sample_match_id
).sort("snapshot_date")

Sample match_id: 12124130


match_id,current_score,display_score,period_1,period_2,period_3,period_4,period_5,period_1_tie_break,period_2_tie_break,period_3_tie_break,period_4_tie_break,period_5_tie_break,normal_time,snapshot_date
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,date
12124130,3,3,6,6,null,null,null,null,null,null,null,null,null,2024-03-03
12124130,3,2,7,6,null,null,null,null,null,null,null,null,null,2024-03-04
12124130,2,2,6,6,null,null,null,null,null,null,null,null,null,2024-03-05


In [23]:
# Sort snapshots by match and snapshot date

home_team_score = home_team_score.sort(
    ["match_id", "snapshot_date"]
)

home_team_score.select(
    ["match_id", "current_score", "display_score",
     "period_1", "period_2", "period_3", "snapshot_date"]
).head(20)

match_id,current_score,display_score,period_1,period_2,period_3,snapshot_date
i64,i64,i64,i64,i64,i64,date
11974049,1,1,null,null,null,2024-02-03
11974049,1,1,null,null,null,2024-02-04
11974049,1,1,null,null,null,2024-02-05
11974052,0,0,null,null,null,2024-02-02
11974052,0,0,null,null,null,2024-02-03
…,…,…,…,…,…,…
11974068,3,3,null,null,null,2024-02-03
11974070,0,0,null,null,null,2024-02-03
11974070,0,0,null,null,null,2024-02-04


In [25]:
# Save the processed home_team_score dataset

processed_path = Path(DATA_ROOT / "Data")

processed_path.mkdir(
    parents=True,
    exist_ok=True
)

home_team_score.write_parquet(
    processed_path / "home_team_score.parquet"
)

In [27]:
# Read the saved file for final validation

check_home_team_score = pl.read_parquet(
    processed_path / "home_team_score.parquet"
)

print("Shape:", check_home_team_score.shape)
print("Columns:", check_home_team_score.columns)

check_home_team_score.head(20)

Shape: (35164, 15)
Columns: ['match_id', 'current_score', 'display_score', 'period_1', 'period_2', 'period_3', 'period_4', 'period_5', 'period_1_tie_break', 'period_2_tie_break', 'period_3_tie_break', 'period_4_tie_break', 'period_5_tie_break', 'normal_time', 'snapshot_date']


match_id,current_score,display_score,period_1,period_2,period_3,period_4,period_5,period_1_tie_break,period_2_tie_break,period_3_tie_break,period_4_tie_break,period_5_tie_break,normal_time,snapshot_date
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,date
11974049,1,1,null,null,null,null,null,null,null,null,null,null,null,2024-02-03
11974049,1,1,null,null,null,null,null,null,null,null,null,null,null,2024-02-04
11974049,1,1,null,null,null,null,null,null,null,null,null,null,null,2024-02-05
11974052,0,0,null,null,null,null,null,null,null,null,null,null,null,2024-02-02
11974052,0,0,null,null,null,null,null,null,null,null,null,null,null,2024-02-03
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
11974068,3,3,null,null,null,null,null,null,null,null,null,null,null,2024-02-03
11974070,0,0,null,null,null,null,null,null,null,null,null,null,null,2024-02-03
11974070,0,0,null,null,null,null,null,null,null,null,null,null,null,2024-02-04
